<a href="https://colab.research.google.com/github/WARRAICH-11/NETSOL/blob/main/Advanced_Python_OOP_Functional_NumPy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Introduction

This notebook covers three powerful areas of Python that are essential for AI/ML development. By the end of this notebook, you should be comfortable with:

* **OOP Deep Dive**: Dunder (magic) methods, class vs instance variables, inheritance, `@property`, `@staticmethod`, `@classmethod`
* **Functional Python**: `map`, `filter`, `reduce`, decorators, generators, `*args`/`**kwargs`
* **NumPy**: Array creation, indexing, slicing, broadcasting, and array math

Please make sure to run <span style="color: red;">all cells</span> regardless of your experience level. Each section builds on the previous one, so read the explanations carefully before attempting the tasks.


## Section 1: Object Oriented Programming (OOP) — Deep Dive

### 1.1 Dunder (Magic) Methods

You already know `__init__`. Python classes support many more **special methods** (also called *dunder methods* because they have **d**ouble **under**scores on both sides). These let your objects behave like built-in Python types — you can make them printable, addable, comparable, and more.

Here is a `Vector` class that demonstrates the most commonly used dunder methods:

In [1]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        # Called by print() — meant for end users # Vector(1, 2)
        return f"Vector({self.x}, {self.y})"

    def __repr__(self):
        # Called in the shell/debugger — meant for developers
        return f"Vector(x={self.x}, y={self.y})"

    def __len__(self):
        # Called by len() — a 2D vector always has 2 components
        return 2

    def __add__(self, other):
        # Called when you write v1 + v2
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar):
        # Called when you write v * 3
        return Vector(self.x * scalar, self.y * scalar)

    def __eq__(self, other):
        # Called when you write v1 == v2
        return self.x == other.x and self.y == other.y


v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1)           # Prints "Vector(1, 2)"
print(repr(v1))     # Prints "Vector(x=1, y=2)"
print(len(v1))      # Prints "2"
print(v1 + v2)      # Prints "Vector(4, 6)"
print(v1 * 3)       # Prints "Vector(3, 6)"
print(v1 == v2)     # Prints "False"


Vector(1, 2)
Vector(x=1, y=2)
2
Vector(4, 6)
Vector(3, 6)
False


### 1.2 Class Variables vs Instance Variables

- **Instance variables** (`self.x`) belong to each individual object — every object has its own copy.
- **Class variables** (defined directly in the class body) are shared across **all** instances of the class.

This distinction matters a lot in ML when you want to track something globally across all objects (e.g., number of models trained).

In [2]:
class Student:
    school = "NIAI"         # class variable — shared by ALL students

    def __init__(self, name, grade):
        self.name = name    # instance variable — unique to each student
        self.grade = grade  # instance variable — unique to each student

s1 = Student("Ali", "A")
s2 = Student("Sara", "B")

print(s1.school)    # Prints "NIAI"
print(s2.school)    # Prints "NIAI"

# Changing the class variable affects ALL instances
Student.school = "NETSOL Institute of AI"
print(s1.school)    # Prints "NETSOL Institute of AI"
print(s2.school)    # Prints "NETSOL Institute of AI"

# But changing it on one instance only affects that instance
s1.school = "FCC"
print(s1.school)    # Prints "FCC"   (instance variable shadows class variable)
print(s2.school)    # Prints "NETSOL Institute of AI"  (still uses class variable)


NIAI
NIAI
NETSOL Institute of AI
NETSOL Institute of AI
FCC
NETSOL Institute of AI


### 1.3 @property, @staticmethod, @classmethod

Python provides three special decorators for class methods:

| Decorator | Receives | Use case |
|---|---|---|
| `@property` | `self` | Make a method look like an attribute; add validation |
| `@staticmethod` | nothing | Utility function that belongs logically to the class |
| `@classmethod` | `cls` (the class) | Factory methods — alternative ways to create instances |


In [3]:
class Circle:
    pi = 3.14159    # class variable

    def __init__(self, radius):
        self._radius = radius   # _ prefix = "private by convention"

    @property
    def radius(self):
        return self._radius     # access as c.radius, not c.radius()

    @radius.setter
    def radius(self, value):
        if value < 0:
            raise ValueError("Radius cannot be negative")
        self._radius = value    # validates input before setting

    @property
    def area(self):
        return Circle.pi * self._radius ** 2   # computed on the fly

    @property
    def circumference(self):
        return 2 * Circle.pi * self._radius

    @staticmethod
    def description():
        # No access to self or cls — pure utility
        return "A circle is defined by its radius."

    @classmethod
    def unit_circle(cls):
        # Factory method — creates a Circle with radius = 1
        return cls(1)


c = Circle(5)
print(c.radius)           # Prints "5"
print(c.area)             # Prints "78.53975"
print(c.circumference)    # Prints "31.4159"
print(Circle.description())      # Prints "A circle is defined by its radius."

unit = Circle.unit_circle()
print(unit.radius)        # Prints "1"
print(unit.area)          # Prints "3.14159"

# Setter with validation
c.radius = 10
print(c.radius)           # Prints "10"
# c.radius = -1           # Would raise ValueError


5
78.53975
31.4159
A circle is defined by its radius.
1
3.14159
10


### 1.4 Inheritance and Method Overriding

**Inheritance** lets one class reuse the code of another. The child class gets all the parent's methods and can:
- **Override** them (replace with its own version)
- **Extend** them (call `super()` and add behaviour)

This is the basis of **polymorphism** — you can write code that works on any `Animal` without knowing if it's a `Dog` or `Cat`.

In [4]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return f"{self.name} makes a sound."

    def __str__(self):
        return f"Animal: {self.name}"


class Dog(Animal):
    def speak(self):              # overrides parent method
        return f"{self.name} says Woof!"


class Cat(Animal):
    def speak(self):
        return f"{self.name} says Meow!"


class GuideDog(Dog):              # multi-level: GuideDog -> Dog -> Animal
    def __init__(self, name, owner):
        super().__init__(name)    # calls Dog.__init__ -> Animal.__init__
        self.owner = owner

    def __str__(self):
        return f"GuideDog: {self.name}, Owner: {self.owner}"


# Polymorphism: same code works for all Animal types
animals = [Dog("Rex"), Cat("Whiskers"), GuideDog("Buddy", "Ahmad")]
for animal in animals:
    print(animal.speak())

print()
print(animals[2])     # Uses GuideDog's __str__


Rex says Woof!
Whiskers says Meow!
Buddy says Woof!

GuideDog: Buddy, Owner: Ahmad


---
## OOP Tasks

### Q1

Create a class `BankAccount` with the following:

- **Instance variables**: `owner` (str), `balance` (float)
- `deposit(amount)` method — adds to balance
- `withdraw(amount)` method — subtracts from balance; raises `ValueError` if insufficient funds
- `__str__` — returns `"Account[owner] | Balance: X"`
- A `@property` called `is_rich` that returns `True` if balance > 100,000
- A `@classmethod` called `zero_account(cls, owner)` that creates an account with balance = 0


In [5]:
class BankAccount:
    def __init__(self, owner: str, balance: float):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float):
        self.balance += amount

    def withdraw(self, amount: float):
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount

    def __str__(self):
        return f"Account[{self.owner}] | Balance: {self.balance}"

    @property
    def is_rich(self):
        return self.balance > 100000

    @classmethod
    def zero_account(cls, owner: str):
        return cls(owner, 0.0)

### Q2

Create a class `Matrix` that:

- Stores a 2D list in `__init__(self, data)`
- `__str__` prints it row by row (one row per line)
- `__add__` adds two matrices element-wise — raise `ValueError` if shapes don't match
- `__mul__` supports scalar multiplication (e.g., `m * 3`)
- `shape` property that returns `(rows, cols)`
- `@staticmethod` called `identity(n)` that returns an n×n identity matrix as a `Matrix` object


In [6]:
class Matrix:
    def __init__(self, data):
        self.data = data

    @property
    def shape(self):
        return (len(self.data), len(self.data[0]))

    def __str__(self):
        return "\n".join(str(row) for row in self.data)

    def __add__(self, other):
        if self.shape != other.shape:
            raise ValueError("Matrix dimensions must match for addition")
        new_data = [
            [a + b for a, b in zip(r1, r2)]
            for r1, r2 in zip(self.data, other.data)
        ]
        return Matrix(new_data)

    def __mul__(self, scalar):
        new_data = [[val * scalar for val in row] for row in self.data]
        return Matrix(new_data)

    @staticmethod
    def identity(n):
        return Matrix([[1 if i == j else 0 for j in range(n)] for i in range(n)])

### Q3

Build a shape hierarchy:

- `Shape` base class with a `@property` called `area` that raises `NotImplementedError`
- `Rectangle(Shape)` — takes `width` and `height`
- `Circle(Shape)` — takes `radius` (use `pi = 3.14159`)
- `Triangle(Shape)` — takes `base` and `height`
- All three override `area`
- Add `__gt__`, `__lt__`, `__eq__` to `Shape` so shapes can be compared and sorted by area
- Add `__str__` to each subclass showing its type and area rounded to 2 decimal places


In [7]:
class Shape:
    @property
    def area(self):
        raise NotImplementedError

    def __eq__(self, other):
        return self.area == other.area

    def __lt__(self, other):
        return self.area < other.area

    def __gt__(self, other):
        return self.area > other.area


class Rectangle(Shape):
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

    def __str__(self):
        return f"Rectangle | Area: {self.area:.2f}"


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return 3.14159 * (self.radius ** 2)

    def __str__(self):
        return f"Circle | Area: {self.area:.2f}"


class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height

    @property
    def area(self):
        return 0.5 * self.base * self.height

    def __str__(self):
        return f"Triangle | Area: {self.area:.2f}"

---
## Section 2: Functional Python

### 2.1 map, filter, reduce

These three functions let you process collections in a clean, expressive style — without writing explicit `for` loops.

- `map(func, iterable)` — apply `func` to every element
- `filter(func, iterable)` — keep only elements where `func` returns `True`
- `reduce(func, iterable)` — combine all elements into a single value (needs `from functools import reduce`)


In [8]:
from functools import reduce

numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# map: square every number
squares = list(map(lambda x: x**2, numbers))
print("Squares:", squares)
# [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]

# filter: keep only even numbers
evens = list(filter(lambda x: x % 2 == 0, numbers))
print("Evens:", evens)
# [2, 4, 6, 8, 10]

# reduce: sum all numbers
total = reduce(lambda acc, x: acc + x, numbers)
print("Total:", total)
# 55

# Chaining all three: square the evens, then sum them
result = reduce(lambda acc, x: acc + x,map(lambda x: x**2,filter(lambda x: x % 2 == 0, numbers)))
print("Sum of squares of evens:", result)
# 4 + 16 + 36 + 64 + 100 = 220


Squares: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Evens: [2, 4, 6, 8, 10]
Total: 55
Sum of squares of evens: 220


### 2.2 *args and **kwargs

- `*args` lets a function accept **any number of positional arguments** — they come in as a tuple
- `**kwargs` lets a function accept **any number of keyword arguments** — they come in as a dict

This is how functions like `print()` can take unlimited arguments.


In [9]:
# *args — any number of positional arguments
def add_all(*args):
    print(f"args received: {args}")
    return sum(args)

print(add_all(1, 2, 3))         # 6
print(add_all(1, 2, 3, 4, 5))   # 15

print()

# **kwargs — any number of keyword arguments
def print_profile(**kwargs):
    for key, value in kwargs.items():
        print(f"  {key}: {value}")

print_profile(name="Ali", age=25, city="Lahore", role="Trainee")

print()

# Combining: fixed args + *args + **kwargs
def mixed(a, b, *args, **kwargs):
    print(f"Required: a={a}, b={b}")
    print(f"Extra positional: {args}")
    print(f"Extra keyword: {kwargs}")

mixed(1, 2, 3, 4, x=10, y=20)


args received: (1, 2, 3)
6
args received: (1, 2, 3, 4, 5)
15

  name: Ali
  age: 25
  city: Lahore
  role: Trainee

Required: a=1, b=2
Extra positional: (3, 4)
Extra keyword: {'x': 10, 'y': 20}


### 2.3 Decorators

A **decorator** is a function that **wraps another function** to add behaviour — without modifying the original function's code.

Think of it like a wrapper around a gift: the gift (your function) stays the same, but the wrapper (decorator) adds something extra — like timing, logging, or validation.

The `@decorator_name` syntax is just shorthand for `func = decorator(func)`.


In [10]:
import time

# ── Basic decorator: measures how long a function takes ──
def timer(func):
    def wrapper(*args, **kwargs):         # *args/**kwargs so it works on ANY function
        start = time.time()
        result = func(*args, **kwargs)    # call the original function
        end = time.time()
        print(f"{func.__name__} took {end - start:.4f} seconds")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

slow_sum(1_000_000)   # slow_sum took ~0.03 seconds

print()

# ── Decorator with arguments: repeat a function N times ──
def repeat(n):
    def decorator(func):
        def wrapper(*args, **kwargs):
            result = None
            for _ in range(n):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(3)
def greet(name):
    print(f"Hello, {name}!")

greet("Ali")    # prints "Hello, Ali!" 3 times


slow_sum took 0.0221 seconds

Hello, Ali!
Hello, Ali!
Hello, Ali!


### 2.4 Generators

A **generator** produces values **one at a time** using the `yield` keyword. Unlike a list, it doesn't compute and store everything in memory upfront — it generates each value only when asked.

This makes generators extremely useful when working with **large datasets** in ML (streaming batches, lazy loading, etc.).


In [11]:
# Regular function: builds the entire list in memory first
def squares_list(n):
    return [x**2 for x in range(n)]

# Generator: produces one value at a time — memory efficient
def squares_gen(n):
    for x in range(n):
        yield x**2      # pauses here; resumes on next call to next()

# Using next() manually
gen = squares_gen(5)
print(next(gen))   # 0
print(next(gen))   # 1
print(next(gen))   # 4

print()

# Or just loop over it
for val in squares_gen(5):
    print(val, end=" ")
print()

print()

# Generator expression (lazy list comprehension)
gen_expr = (x**2 for x in range(10))
print(list(gen_expr))   # [0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

# Practical example: batch generator for ML training
def batch_generator(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

dataset = list(range(20))
for batch in batch_generator(dataset, 5):
    print("Batch:", batch)


0
1
4

0 1 4 9 16 

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
Batch: [0, 1, 2, 3, 4]
Batch: [5, 6, 7, 8, 9]
Batch: [10, 11, 12, 13, 14]
Batch: [15, 16, 17, 18, 19]


---
## Functional Python Tasks

### Q4

Write a function `pipeline(data, *funcs)` that takes a list and any number of functions, and applies them **in sequence** using `map` or `filter`. Return the final result as a list.

- If the function returns a boolean (i.e., it's a filter), use `filter`
- If it transforms values, use `map`

**Hint**: you can check a function's behavior by testing it on a sample value — if the result is `True`/`False`, it's a filter.


In [12]:
def pipeline(data, *funcs):
    result = data
    for func in funcs:
        # Check if the function acts as a filter (returns boolean on test element)
        if isinstance(func(result[0]), bool):
            result = list(filter(func, result))
        else:
            result = list(map(func, result))
    return result

### Q5

Write a decorator `@validate_positive` that raises a `ValueError` if **any argument** passed to the decorated function is negative or zero.

The decorator should work on functions with any number of arguments.


In [13]:
def validate_positive(func):
    def wrapper(*args, **kwargs):
        all_vals = list(args) + list(kwargs.values())
        if any(v <= 0 for v in all_vals if isinstance(v, (int, float))):
            raise ValueError("All arguments must be positive numbers (> 0)")
        return func(*args, **kwargs)
    return wrapper

### Q6

Write a **generator function** `running_stats(numbers)` that yields a tuple `(mean, variance)` after each new number is added to the stream.

**Important constraint**: do NOT store the full list or recompute from scratch each time. Update mean and variance **incrementally** using Welford's online algorithm or a running sum approach.

This is exactly how streaming ML systems compute statistics on live data.


In [14]:
def running_stats(numbers):
    n = 0
    mean = 0.0
    M2 = 0.0

    for x in numbers:
        n += 1
        delta = x - mean
        mean += delta / n
        delta2 = x - mean
        M2 += delta * delta2

        variance = 0.0 if n < 2 else M2 / n  # Population variance (or M2 / (n - 1) for sample variance)
        yield mean, variance

### Q7

Using **only** `map`, `filter`, and `reduce` (no `for` or `while` loops), do the following in one pipeline:

1. Take the list of sentences below
2. Filter sentences that have **more than 5 words**
3. Convert each to **title case**
4. Concatenate all into one string separated by `" | "`


In [15]:
from functools import reduce

sentences = [
    "the quick brown fox jumps over the lazy dog",
    "hello world",
    "artificial intelligence is transforming the world today",
    "python",
    "machine learning requires a lot of data and compute"
]

result = reduce(
    lambda acc, x: f"{acc} | {x}",
    map(
        lambda s: s.title(),
        filter(lambda s: len(s.split()) > 5, sentences)
    )
)

---
## Section 3: NumPy

### 3.1 Creating Arrays

NumPy is the backbone of almost every ML library (TensorFlow, PyTorch, scikit-learn all use NumPy arrays internally). Its core object is the `ndarray` — an n-dimensional array that supports fast, vectorized operations.

Key advantage over Python lists: NumPy operations run in **compiled C code**, making them 10–100x faster than equivalent Python loops.


In [16]:
import numpy as np

# From a Python list
a = np.array([1, 2, 3, 4, 5])
print("1D array:", a)
print("dtype:", a.dtype)      # int64
print("shape:", a.shape)      # (5,)

print()

# 2D array (matrix) — rows x columns
b = np.array([[1, 2, 3],
              [4, 5, 6]])
print("2D array shape:", b.shape)   # (2, 3)

print()

# Useful constructors
print("zeros:\n",   np.zeros((3, 3)))
print("ones:\n",    np.ones((2, 4)))
print("identity:\n", np.eye(3))
print("arange:", np.arange(0, 10, 2))        # [0 2 4 6 8]
print("linspace:", np.linspace(0, 1, 5))     # [0. 0.25 0.5 0.75 1.]

np.random.seed(42)
print("random normal:\n", np.random.randn(3, 3).round(2))


1D array: [1 2 3 4 5]
dtype: int64
shape: (5,)

2D array shape: (2, 3)

zeros:
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
ones:
 [[1. 1. 1. 1.]
 [1. 1. 1. 1.]]
identity:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
arange: [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
random normal:
 [[ 0.5  -0.14  0.65]
 [ 1.52 -0.23 -0.23]
 [ 1.58  0.77 -0.47]]


### 3.2 Indexing and Slicing

NumPy indexing uses `[row, col]` notation for 2D arrays. You can also use **boolean masks** to select elements based on a condition — this is used constantly in data preprocessing.


In [17]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

print("Element [0,1]:", a[0, 1])       # 2  — row 0, col 1
print("Column 1:", a[:, 1])            # [2 5 8] — all rows, col 1
print("Row 1:", a[1, :])               # [4 5 6] — row 1, all cols
print("Submatrix:\n", a[0:2, 0:2])   # [[1 2], [4 5]]

print()

# Boolean masking — returns elements where condition is True
print("Elements > 5:", a[a > 5])       # [6 7 8 9]

# Set all even elements to 0
b = a.copy()
b[b % 2 == 0] = 0
print("Even elements zeroed:\n", b)

print()

# Fancy indexing — select specific rows
print("Rows 0 and 2:\n", a[[0, 2]])


Element [0,1]: 2
Column 1: [2 5 8]
Row 1: [4 5 6]
Submatrix:
 [[1 2]
 [4 5]]

Elements > 5: [6 7 8 9]
Even elements zeroed:
 [[1 0 3]
 [0 5 0]
 [7 0 9]]

Rows 0 and 2:
 [[1 2 3]
 [7 8 9]]


### 3.3 Array Math and Broadcasting

NumPy operations are **element-wise by default** — no loops needed.

**Broadcasting** is NumPy's way of performing operations between arrays of different shapes. The smaller array is automatically "stretched" to match the larger one. This is heavily used in ML — for example, subtracting the mean from each row of a dataset.


In [18]:
import numpy as np

x = np.array([1, 2, 3])
y = np.array([4, 5, 6])

# Element-wise operations
print("x + y =", x + y)        # [5 7 9]
print("x * y =", x * y)        # [4 10 18]
print("x ** 2 =", x ** 2)      # [1 4 9]

# Dot product
print("dot(x,y) =", np.dot(x, y))   # 1*4 + 2*5 + 3*6 = 32

print()

# Matrix multiplication
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print("A @ B =\n", A @ B)     # [[19 22], [43 50]]

print()

# Broadcasting: add a 1D array to each row of a 2D array
data = np.array([[1, 2, 3],
                 [4, 5, 6]])        # shape (2, 3)
bias = np.array([10, 20, 30])       # shape (3,) — broadcast across rows

print("data + bias =\n", data + bias)   # [[11 22 33], [14 25 36]]

# Broadcasting: subtract column mean from each column (normalization)
col_means = data.mean(axis=0)            # shape (3,) — mean of each column
print("column means:", col_means)
print("centered:\n", data - col_means)


x + y = [5 7 9]
x * y = [ 4 10 18]
x ** 2 = [1 4 9]
dot(x,y) = 32

A @ B =
 [[19 22]
 [43 50]]

data + bias =
 [[11 22 33]
 [14 25 36]]
column means: [2.5 3.5 4.5]
centered:
 [[-1.5 -1.5 -1.5]
 [ 1.5  1.5  1.5]]


### 3.4 Useful Aggregate Operations

These operations are used constantly in ML — computing loss functions, accuracy, normalization, etc.


In [19]:
import numpy as np

a = np.array([[1, 2, 3],
              [4, 5, 6]])

print("Sum (all):", a.sum())           # 21
print("Sum (per col):", a.sum(axis=0)) # [5 7 9]
print("Sum (per row):", a.sum(axis=1)) # [6 15]
print("Mean:", a.mean())               # 3.5
print("Std:", a.std().round(4))        # 1.7078
print("Max:", a.max())                 # 6
print("Min:", a.min())                 # 1
print("Argmax:", a.argmax())           # 5 (flat index of max element)
print("Argmax per row:", a.argmax(axis=1))  # [2 2] — col index of max in each row

print()
print("Reshape to (3,2):\n", a.reshape(3, 2))
print("Transpose:\n", a.T)
print("Flatten:", a.flatten())


Sum (all): 21
Sum (per col): [5 7 9]
Sum (per row): [ 6 15]
Mean: 3.5
Std: 1.7078
Max: 6
Min: 1
Argmax: 5
Argmax per row: [2 2]

Reshape to (3,2):
 [[1 2]
 [3 4]
 [5 6]]
Transpose:
 [[1 4]
 [2 5]
 [3 6]]
Flatten: [1 2 3 4 5 6]


---
## NumPy Tasks

### Q8

Given the 2D NumPy array `scores` below (rows = students, columns = subjects), write NumPy code (no loops) to:

1. Find the **mean score per student** (one value per student)
2. Find the **index of the top scoring student** (highest mean)
3. **Normalize** all scores to range [0, 1] using min-max normalization: `(x - min) / (max - min)`
4. Count how many students scored **above 70 in every subject**


In [21]:
import numpy as np

# Sample dataset (Students x Subjects)
scores = np.array([
    [85, 90, 78],
    [60, 65, 70],
    [92, 88, 95],
    [75, 80, 72]
])

# 1. Average score per student
student_means = scores.mean(axis=1)

# 2. Index of top scoring student (highest average across subjects)
top_student_idx = student_means.argmax()

# 3. Normalize scores to [0, 1]
normalized_scores = (scores - scores.min()) / (scores.max() - scores.min())

# 4. Students who scored above 70 in ALL subjects
all_above_70 = np.where((scores > 70).all(axis=1))[0]

### Q9

Implement the following three functions using **only NumPy** — no Python loops:

1. `cosine_similarity(a, b)` — cosine similarity between two 1D vectors: `dot(a,b) / (||a|| * ||b||)`
2. `softmax(x)` — converts a 1D array to probabilities that sum to 1: `exp(x) / sum(exp(x))`
3. `batch_normalize(X)` — normalize each **column** of a 2D array to have mean=0, std=1

These three functions are used directly in neural networks and ML pipelines.


In [ ]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def softmax(x):
    e_x = np.exp(x - np.max(x))  # Shift for numerical stability
    return e_x / np.sum(e_x)

def batch_normalize(X):
    mean = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    # Add small epsilon to prevent division by zero
    return (X - mean) / (std + 1e-8)